# Adverse impact: impact ratio, significance and power

> **Scope.** Run this only against models and apps you own or the toy targets defined here. No third-party systems, no real people's data. Everything in this repository is synthetic.

Enter selection counts from your own audit (for example the ML scorer lab's CSV).

In [ ]:
import math
from scipy import stats

sel_focal, n_focal = 18, 60   # replace with your counts
sel_ref, n_ref = 40, 90

r_f, r_r = sel_focal / n_focal, sel_ref / n_ref
print(f"selection rates: focal {r_f:.3f}, reference {r_r:.3f}, impact ratio {r_f / r_r:.3f}")
p = (sel_focal + sel_ref) / (n_focal + n_ref)
z = (r_r - r_f) / math.sqrt(p * (1 - p) * (1 / n_focal + 1 / n_ref))
print(f"two-proportion z = {z:.2f}, two-sided p = {2 * (1 - stats.norm.cdf(abs(z))):.4f}")
print("Fisher exact p =", stats.fisher_exact([[sel_focal, n_focal - sel_focal], [sel_ref, n_ref - sel_ref]]).pvalue)

In [ ]:
def power_two_prop(p1, p2, n1, n2, alpha=0.05):
    za = stats.norm.ppf(1 - alpha / 2)
    pbar = (p1 * n1 + p2 * n2) / (n1 + n2)
    se0 = math.sqrt(pbar * (1 - pbar) * (1 / n1 + 1 / n2))
    se1 = math.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
    d = abs(p1 - p2)
    return stats.norm.cdf((d - za * se0) / se1) + stats.norm.cdf((-d - za * se0) / se1)

for n in (20, 50, 100, 200, 400):
    print(n, round(power_two_prop(0.30, 0.45, n, n), 3))